# CrewAI: Building Multi-Agent Systems

This notebook provides a comprehensive, practical guide to CrewAI — a framework for orchestrating role-playing autonomous AI agents that work together to complete complex tasks.

All examples use **GPT-4o-mini** as the language model and **text-embedding-3-small** for embedding operations.

---

## Table of Contents

1. Installation and Environment Setup
2. Core Concepts: Agent, Task, Crew
3. Agent Properties Deep Dive
4. Task Properties Deep Dive
5. Crew Properties and Process Types
6. Built-in Tools Overview
7. Implementation 1 — Research and Report Writing Crew
8. Implementation 2 — Software Development Crew
9. Implementation 3 — Customer Support Triage Crew
10. Implementation 4 — Financial Analysis Crew with Memory
11. Implementation 5 — RAG-Powered Knowledge Base Crew
12. Advanced: Custom Tools
13. Advanced: Human-in-the-Loop
14. Advanced: Async Execution and Callbacks
15. Best Practices and Common Pitfalls

---
## 1. Installation and Environment Setup

In [43]:
# Install required packages
%pip install crewai crewai-tools openai langchain-openai python-dotenv --quiet

In [57]:
import os
from google.colab import userdata
from crewai.memory import Memory

# Keys
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["SERPER_API_KEY"] = userdata.get("SERPER_API_KEY")

# Force correct embedding model
memory = Memory(
    embedder={
        "provider": "openai",
        "config": {"model": "text-embedding-3-small"}
    }
)

In [45]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Define the LLM used across all agents in this notebook
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,           # Lower temperature for more consistent, professional output
    max_tokens=2048,
    api_key=OPENAI_API_KEY # Explicitly pass the API key
)


# Define the embedding model for memory and RAG features
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY # Explicitly pass the API key
)

print(f"LLM: {llm.model_name}")
print(f"Embeddings: {embeddings.model}")

LLM: gpt-4o-mini
Embeddings: text-embedding-3-small


---
## 2. Core Concepts: Agent, Task, Crew

CrewAI is built around three primary primitives:

| Primitive | Purpose |
|-----------|-------------------------------------------|
| **Agent** | An autonomous entity with a role, goal, and backstory. It uses tools and an LLM to reason and act. |
| **Task** | A discrete unit of work assigned to one or more agents, with a clear expected output. |
| **Crew** | An orchestrator that manages a collection of agents and tasks, defines the execution process, and aggregates results. |

The typical workflow:
1. Define your agents with specific roles.
2. Define tasks and assign them to agents.
3. Assemble a crew and run it.


---
## 3. Agent Properties Deep Dive

Below is a minimal agent followed by a fully configured agent with explanations for every property.

In [46]:
from crewai import Agent

minimal_agent = Agent(
    role="Data Analyst",
    goal="Extract actionable insights from raw datasets.",
    backstory=(
        "You are a senior data analyst with ten years of experience "
        "working in the financial services industry."
    ),
    llm="gpt-4o-mini",
)

print("Minimal agent created:", minimal_agent.role)

Minimal agent created: Data Analyst


In [47]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# --- Fully Configured Agent ---
fully_configured_agent = Agent(
    # ----------------------------------------------------------------
    # IDENTITY PROPERTIES
    # ----------------------------------------------------------------
    role="Senior Market Research Analyst",
    # role: A short label that defines who this agent is.
    # It influences how the LLM frames its reasoning and responses.

    goal=(
        "Produce accurate, evidence-based market research reports "
        "that help executives make high-confidence investment decisions."
    ),
    # goal: What this agent is trying to achieve. The LLM uses this
    # as an objective to optimize toward throughout task execution.

    backstory=(
        "You spent 15 years as a research analyst at top-tier consulting "
        "firms including McKinsey and BCG. You have deep expertise in "
        "technology, healthcare, and consumer sectors. Your reports are "
        "known for their precision and depth."
    ),
    # backstory: A rich narrative that shapes the agent's persona.
    # The more detailed and realistic, the better the output quality.

    # ----------------------------------------------------------------
    # LLM AND TOOL PROPERTIES
    # ----------------------------------------------------------------
   llm="gpt-4o-mini",
    # llm: The language model powering this agent.
    # Each agent can have a different LLM if needed.

    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    # tools: A list of Tool objects the agent can invoke.
    # Tools give agents the ability to take actions beyond text generation.

    # ----------------------------------------------------------------
    # BEHAVIOR PROPERTIES
    # ----------------------------------------------------------------
    verbose=True,
    # verbose: When True, prints the agent's internal reasoning chain
    # (Thought -> Action -> Observation loop) to stdout. Useful for debugging.

    allow_delegation=False,
    # allow_delegation: When True, this agent can delegate subtasks
    # to other agents in the crew. Set to False to keep it focused.

    max_iter=10,
    # max_iter: Maximum number of reasoning iterations before the agent
    # is forced to produce a final answer. Prevents infinite loops.

    max_rpm=20,
    # max_rpm: Maximum API requests per minute. Useful for rate-limit management.

    memory=True,
    # memory: When True, the agent retains context across task interactions
    # within the same crew run. Powered by the embedding model.

    # ----------------------------------------------------------------
    # OPTIONAL: CUSTOM SYSTEM PROMPT
    # ----------------------------------------------------------------
    # system_template: Override the default system prompt entirely.
    # Use {role}, {goal}, {backstory} as placeholders.
    # system_template="You are {role}. Your goal: {goal}. Background: {backstory}.",
)

print("Fully configured agent created:", fully_configured_agent.role)

Fully configured agent created: Senior Market Research Analyst


### Agent Property Summary

| Property | Type | Required | Description |
|---|---|---|---|
| `role` | str | Yes | The agent's job title / persona label |
| `goal` | str | Yes | What the agent aims to accomplish |
| `backstory` | str | Yes | Rich narrative that shapes behavior |
| `llm` | LLM | No | Language model (defaults to OpenAI GPT-4) |
| `tools` | list | No | Tool objects the agent can call |
| `verbose` | bool | No | Print reasoning steps (default: False) |
| `allow_delegation` | bool | No | Allow task hand-off to other agents |
| `max_iter` | int | No | Max reasoning loops (default: 15) |
| `max_rpm` | int | No | Rate limit for API calls |
| `memory` | bool | No | Enable cross-task memory |
| `system_template` | str | No | Custom system prompt template |

---
## 4. Task Properties Deep Dive

In [48]:
from crewai import Task
from pydantic import BaseModel
from typing import List

# Define a Pydantic model for structured output
class MarketReport(BaseModel):
    company_name: str
    market_size_usd_billion: float
    key_competitors: List[str]
    growth_rate_percent: float
    recommendation: str

# --- Minimal Task ---
minimal_task = Task(
    description="Summarize the current state of the electric vehicle market.",
    expected_output="A 300-word summary covering market size, key players, and growth trends.",
    agent=minimal_agent,
)

# --- Fully Configured Task ---
research_task = Task(
    # ----------------------------------------------------------------
    # CORE PROPERTIES
    # ----------------------------------------------------------------
    description=(
        "Conduct a comprehensive market analysis for {company_name} "
        "operating in the {industry} sector. "
        "Focus on: (1) total addressable market size, "
        "(2) top 5 competitors, (3) annual growth rate, "
        "(4) a strategic recommendation for market entry."
    ),
    # description: The full instruction for the task. Supports
    # {variable} placeholders that get filled in at crew kickoff.

    expected_output=(
        "A structured JSON report containing: company_name, "
        "market_size_usd_billion, key_competitors (list of 5), "
        "growth_rate_percent, and recommendation."
    ),
    # expected_output: Describes what a successful completion looks like.
    # The LLM uses this to self-evaluate and format its response.

    agent=fully_configured_agent,
    # agent: The agent responsible for this task.

    # ----------------------------------------------------------------
    # STRUCTURED OUTPUT
    # ----------------------------------------------------------------
    output_pydantic=MarketReport,
    # output_pydantic: Force the output to conform to a Pydantic model.
    # CrewAI will validate and parse the output automatically.

    # output_json=MarketReport,  # Alternative: output as a plain dict
    # output_file="report.md",   # Alternative: save output to a file

    # ----------------------------------------------------------------
    # DEPENDENCY PROPERTIES
    # ----------------------------------------------------------------
    # context=[another_task],
    # context: A list of tasks whose outputs are passed as context
    # to this task. Enables data flow between sequential steps.

    # ----------------------------------------------------------------
    # CALLBACK
    # ----------------------------------------------------------------
    # callback=my_callback_function,
    # callback: A Python function called with the task output
    # after the task completes. Useful for logging or downstream actions.

    # ----------------------------------------------------------------
    # HUMAN INPUT
    # ----------------------------------------------------------------
    human_input=False,
    # human_input: When True, the agent pauses and requests human
    # review/approval before finalizing the output.
)

print("Tasks created successfully.")

Tasks created successfully.


### Task Property Summary

| Property | Type | Required | Description |
|---|---|---|---|
| `description` | str | Yes | What needs to be done (supports `{placeholders}`) |
| `expected_output` | str | Yes | What a completed result looks like |
| `agent` | Agent | Yes | The responsible agent |
| `context` | list[Task] | No | Upstream tasks whose output feeds this one |
| `output_pydantic` | BaseModel | No | Validate output against a Pydantic model |
| `output_json` | BaseModel | No | Return output as a plain dict |
| `output_file` | str | No | Write output to a file path |
| `callback` | callable | No | Function called on task completion |
| `human_input` | bool | No | Pause for human review before finalizing |

---
## 5. Crew Properties and Process Types

In [49]:
from crewai import Crew, Process

# Process.sequential  — tasks run one after another in order
# Process.hierarchical — a manager agent orchestrates which agent does what

# --- Sequential Crew (most common) ---
sequential_crew = Crew(
    agents=[minimal_agent],
    tasks=[minimal_task],

    process=Process.sequential,
    # process: Defines the execution strategy.
    # Sequential: task[0] -> task[1] -> task[2] ...
    # Hierarchical: a manager agent delegates to worker agents.

    verbose=True,
    # verbose: Print crew-level orchestration logs.

    memory=False,
    # memory: Enable short-term, long-term, and entity memory
    # across the entire crew. Requires an embedding model.

    embedder={
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small"
        }
    },
    # embedder: Configuration for the embedding model used in memory.

    max_rpm=30,
    # max_rpm: Crew-level rate limit (applies across all agents).

    # share_crew=False,
    # share_crew: If True, shares crew metadata with CrewAI
    # for performance benchmarking (opt-in).

    # step_callback=my_step_fn,
    # step_callback: Called after every agent action step.

    # task_callback=my_task_fn,
    # task_callback: Called after every task completes.
)

print("Sequential crew assembled.")

Sequential crew assembled.


In [50]:
from crewai import Agent, Task, Crew, Process

# ✅ FIX: no trailing comma
llm = "gpt-4o-mini"

worker_agent_1 = Agent(
    role="Web Researcher",
    goal="Find accurate information from online sources.",
    backstory="You are a meticulous researcher who always cites sources.",
    llm=llm,
    verbose=True,
)

worker_agent_2 = Agent(
    role="Content Writer",
    goal="Produce clear, concise, and well-structured written content.",
    backstory="You are a professional business writer with an economics background.",
    llm=llm,
    verbose=True,
)

write_task = Task(
    description="Write a 500-word executive summary on renewable energy trends.",
    expected_output="A professional executive summary suitable for a board presentation.",
    agent=worker_agent_2,
)

hierarchical_crew = Crew(
    agents=[worker_agent_1, worker_agent_2],
    tasks=[write_task],
    process=Process.hierarchical,
    manager_llm=llm,
    verbose=True,
)

print("Hierarchical crew assembled.")

Hierarchical crew assembled.


---
## 6. Built-in Tools Overview

CrewAI ships with a broad library of tools through the `crewai-tools` package.

In [51]:
# Overview of key tools available in crewai-tools

tool_catalog = {
    "SerperDevTool": "Google Search via Serper API — good for finding current web results",
    "ScrapeWebsiteTool": "Scrape full text content from any URL",
    "FileReadTool": "Read content from a local file",
    "FileWriterTool": "Write content to a local file",
    "DirectoryReadTool": "List and read files in a directory",
    "PDFSearchTool": "Semantic search over PDF documents using RAG",
    "CSVSearchTool": "Semantic search over CSV files",
    "TXTSearchTool": "Semantic search over plain text files",
    "JSONSearchTool": "Semantic search over JSON documents",
    "DOCXSearchTool": "Semantic search over Word documents",
    "YoutubeChannelSearchTool": "Search a YouTube channel's transcripts",
    "YoutubeVideoSearchTool": "Search a YouTube video's transcript",
    "GithubSearchTool": "Search GitHub repositories and code",
    "CodeDocsSearchTool": "Semantic search over code documentation",
    "WebsiteSearchTool": "RAG-based semantic search over a website",
    "BrowserbaseLoadTool": "Load pages using a headless browser (for JS-heavy sites)",
    "DallETool": "Generate images using DALL-E",
    "VisionTool": "Analyze images with GPT-4 Vision",
}

print(f"{'Tool Name':<30} {'Description'}")
print("-" * 90)
for name, desc in tool_catalog.items():
    print(f"{name:<30} {desc}")

Tool Name                      Description
------------------------------------------------------------------------------------------
SerperDevTool                  Google Search via Serper API — good for finding current web results
ScrapeWebsiteTool              Scrape full text content from any URL
FileReadTool                   Read content from a local file
FileWriterTool                 Write content to a local file
DirectoryReadTool              List and read files in a directory
PDFSearchTool                  Semantic search over PDF documents using RAG
CSVSearchTool                  Semantic search over CSV files
TXTSearchTool                  Semantic search over plain text files
JSONSearchTool                 Semantic search over JSON documents
DOCXSearchTool                 Semantic search over Word documents
YoutubeChannelSearchTool       Search a YouTube channel's transcripts
YoutubeVideoSearchTool         Search a YouTube video's transcript
GithubSearchTool               

In [55]:
from crewai.memory import Memory

custom_memory = Memory(
    embedder={
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-small"
        }
    }
)

In [58]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# ✅ Use string OR CrewAI LLM (recommended simple way)
llm = "gpt-4o-mini"

# --- Agents ---
researcher = Agent(
    role="Senior Investment Research Analyst",
    goal=(
        "Gather comprehensive, factual information about companies "
        "including financial performance, competitive position, and market trends."
    ),
    backstory=(
        "You are a CFA charterholder with 12 years of experience in equity research "
        "at a leading investment bank. You rely only on verifiable data and cite all sources."
    ),
    llm=llm,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
    allow_delegation=False,
    max_iter=8,
    memory=True,
)

report_writer = Agent(
    role="Business Report Writer",
    goal=(
        "Convert research into clear, structured, executive-ready investment memos."
    ),
    backstory=(
        "You are a former Bloomberg and Wall Street Journal writer. "
        "You follow the Pyramid Principle and write with clarity and precision."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=6,
    memory=True,
)

# --- Tasks ---
research_task = Task(
    description=(
        "Research {company_name} in the {industry} industry.\n\n"
        "Collect:\n"
        "1. Latest revenue and profit figures\n"
        "2. Top 3 competitors with positioning\n"
        "3. Recent news impacting stock price\n"
        "4. Analyst consensus rating\n\n"
        "IMPORTANT: Include source URLs for every fact."
    ),
    expected_output=(
        "Structured bullet-point research brief with clearly labeled sections "
        "and source links for each data point."
    ),
    agent=researcher,
)

writing_task = Task(
    description=(
        "Using the research brief, write a one-page investment memo on {company_name}.\n\n"
        "Structure:\n"
        "- Executive Summary (2 sentences)\n"
        "- Company Overview (1 paragraph)\n"
        "- Financial Highlights (bullets)\n"
        "- Competitive Landscape (1 paragraph)\n"
        "- Risks (3 bullets)\n"
        "- Recommendation (1 paragraph)\n\n"
        "Ensure clarity, conciseness, and professional tone."
    ),
    expected_output=(
        "A polished, executive-ready investment memo in Markdown format."
    ),
    agent=report_writer,
    context=[research_task],
    output_file="investment_memo.md",
)

# --- Crew ---
research_crew = Crew(
    agents=[researcher, report_writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    memory=custom_memory,
    verbose=True,
)

# --- Run ---
result = research_crew.kickoff(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d6a43120-c788-4894-81e7-7ea4a93e016e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research NVIDIA in the semiconductors and AI computing industry.                                         │
│                                                                                                                 │
│  Collect:                                                                                                       │
│  1. Latest revenue and profit figures                                                                           │
│  2. Top 3 competitors with positioning                                                                          │
│  3. Recent news impacting stock price                                                                           │
│  4. Analyst consensus rating                                                                                    │
│                                                                                                                 │
│  IMPORTANT: Include source URLs for every fact.                                                                 │
│  ID: db66ac1e-90c8-4d66-b0a0-0c68d945f738                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ❌ Memory Query Error ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Query Failed                                                                                            │
│  Source: Unified Memory                                                                                         │
│  Error: Error code: 403 - {'error': {'message': 'You are not allowed to generate embeddings from this model',   │
│  'type': 'invalid_request_error', 'param': None, 'code': None}}                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Investment Research Analyst                                                                      │
│                                                                                                                 │
│  Task: Research NVIDIA in the semiconductors and AI computing industry.                                         │
│                                                                                                                 │
│  Collect:                                                                                                       │
│  1. Latest revenue and profit figures                                                                           │
│  2. Top 3 competitors with positioning                                                                          │
│  3. Recent news impacting stock price                                                                           │
│  4. Analyst consensus rating                                                                                    │
│                                                                                                                 │
│  IMPORTANT: Include source URLs for every fact.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#70) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'NVIDIA latest revenue and profit figures 2023'}                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Business Report Writer                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # Investment Memo: NVIDIA Corporation                                                                          │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  NVIDIA has demonstrated exceptional growth with a staggering 125.85% increase in revenue for 2023, reaching    │
│  $26.974 billion, and is projected to continue this trajectory with expected revenues of $60.922 billion in     │
│  2024. Despite recent stock volatility due to competitive pressures and concerns over export restrictions,      │
│  analyst consensus remains strongly bullish on the company’s future prospects.                                  │
│                                                                                                                 │
│  ## Company Overview                                                                                            │
│  NVIDIA Corporation, a leader in GPU technology, specializes in the design of graphics processing units for     │
│  gaming, professional markets, and AI applications. The company's products are heavily integrated into gaming   │
│  PCs, data centers, and automotive solutions, positioning it at the forefront of multiple high-growth segments  │
│  in technology. NVIDIA has effectively capitalized on the rising demand for AI and machine learning             │
│  technologies, reinforcing its market dominance and innovative prowess.                                         │
│                                                                                                                 │
│  ## Financial Highlights                                                                                        │
│  - **Revenue for 2023**: $26.974 billion (125.85% increase from 2022).                                          │
│  - **Projected Revenue for 2024**: $60.922 billion.                                                             │
│  - **Profit for 2023**: $120 billion in net income.                                                             │
│  - **Stock Performance**: Recent 17% decline due to market concerns.                                            │
│  - **Analyst Consensus Rating**: "Strong Buy" from 42 analysts.                                                 │
│  - **Average Target Price**: Approximately $274.34.                                                             │
│                                                                                                                 │
│  ## Competitive Landscape                                                                                       │
│  NVIDIA faces robust competition in the GPU and AI sectors, primarily from AMD, which emphasizes                │
│  cost-effective solutions for AI, and Intel, which is developing AI-specific chips. Additionally, Google        │
│  presents challenges through its Tensor Processing Units tailored for AI workloads. Despite this competitive    │
│  pressure, NVIDIA's established brand and technological advancements in AI position it strongly in the market.  │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the research brief, write a one-page investment memo on NVIDIA.                                    │
│                                                                                                                 │
│  Structure:                                                                                                     │
│  - Executive Summary (2 sentences)                                                                              │
│  - Company Overview (1 paragraph)                                                                               │
│  - Financial Highlights (bullets)                                                                               │
│  - Competitive Landscape (1 paragraph)                                                                          │
│  - Risks (3 bullets)                                                                                            │
│  - Recommendation (1 paragraph)                                                                                 │
│                                                                                                                 │
│  Ensure clarity, conciseness, and professional tone.                                                            │
│  Agent: Business Report Writer                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d6a43120-c788-4894-81e7-7ea4a93e016e                                                                       │
│  Final Output: ```markdown                                                                                      │
│  # Investment Memo: NVIDIA Corporation                                                                          │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  NVIDIA has demonstrated exceptional growth with a staggering 125.85% increase in revenue for 2023, reaching    │
│  $26.974 billion, and is projected to continue this trajectory with expected revenues of $60.922 billion in     │
│  2024. Despite recent stock volatility due to competitive pressures and concerns over export restrictions,      │
│  analyst consensus remains strongly bullish on the company’s future prospects.                                  │
│                                                                                                                 │
│  ## Company Overview                                                                                            │
│  NVIDIA Corporation, a leader in GPU technology, specializes in the design of graphics processing units for     │
│  gaming, professional markets, and AI applications. The company's products are heavily integrated into gaming   │
│  PCs, data centers, and automotive solutions, positioning it at the forefront of multiple high-growth segments  │
│  in technology. NVIDIA has effectively capitalized on the rising demand for AI and machine learning             │
│  technologies, reinforcing its market dominance and innovative prowess.                                         │
│                                                                                                                 │
│  ## Financial Highlights                                                                                        │
│  - **Revenue for 2023**: $26.974 billion (125.85% increase from 2022).                                          │
│  - **Projected Revenue for 2024**: $60.922 billion.                                                             │
│  - **Profit for 2023**: $120 billion in net income.                                                             │
│  - **Stock Performance**: Recent 17% decline due to market concerns.                                            │
│  - **Analyst Consensus Rating**: "Strong Buy" from 42 analysts.                                                 │
│  - **Average Target Price**: Approximately $274.34.                                                             │
│                                                                                                                 │
│  ## Competitive Landscape                                                                                       │
│  NVIDIA faces robust competition in the GPU and AI sectors, primarily from AMD, which emphasizes                │
│  cost-effective solutions for AI, and Intel, which is developing AI-specific chips. Additionally, Google        │
│  presents challenges through its Tensor Processing Units tailored for AI workloads. Despite this competitive    │
│  pressure, NVIDIA's established brand and technological advancements in AI position it strongly in the market.  │
│                                                       


FINAL OUTPUT
```markdown
# Investment Memo: NVIDIA Corporation

## Executive Summary
NVIDIA has demonstrated exceptional growth with a staggering 125.85% increase in revenue for 2023, reaching $26.974 billion, and is projected to continue this trajectory with expected revenues of $60.922 billion in 2024. Despite recent stock volatility due to competitive pressures and concerns over export restrictions, analyst consensus remains strongly bullish on the company’s future prospects.

## Company Overview
NVIDIA Corporation, a leader in GPU technology, specializes in the design of graphics processing units for gaming, professional markets, and AI applications. The company's products are heavily integrated into gaming PCs, data centers, and automotive solutions, positioning it at the forefront of multiple high-growth segments in technology. NVIDIA has effectively capitalized on the rising demand for AI and machine learning technologies, reinforcing its market dominance and innovative prowess



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [73]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

llm = "gpt-4o-mini"

# --- Manager Agent ---
manager = Agent(
    role="Investment Research Manager",
    goal=(
        "Oversee the research and writing process to produce a high-quality "
        "investment memo. Decide task order and delegate effectively."
    ),
    backstory=(
        "You are a senior portfolio manager overseeing analysts and writers. "
        "You break down problems and assign tasks efficiently."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=True,   # 🔥 IMPORTANT
)

# --- Worker Agents ---
researcher = Agent(
    role="Senior Investment Research Analyst",
    goal="Gather comprehensive and factual company data with sources.",
    backstory="CFA with 12 years of equity research experience.",
    llm=llm,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
    allow_delegation=False,
)

report_writer = Agent(
    role="Business Report Writer",
    goal="Write clear, structured investment memos.",
    backstory="Former Bloomberg/WSJ writer.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# --- Tasks (no strict ordering now) ---
research_task = Task(
    description="Research {company_name} in {industry} with financials, competitors, news, ratings. Include sources.",
    expected_output="Bullet-point research brief with sources.",
    agent=researcher,
)

writing_task = Task(
    description="Write a structured investment memo using research.",
    expected_output="Executive-ready memo in Markdown.",
    agent=report_writer,
)

# --- Crew ---
research_crew = Crew(
    agents=[researcher, report_writer],   # ✅ ONLY workers
    tasks=[research_task, writing_task],
    process=Process.hierarchical,
    manager_agent=manager,                # ✅ manager here
    memory=custom_memory,
    verbose=True,
)
# --- Run ---
result = research_crew.kickoff(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 61099a39-3846-4995-a26a-7de26813f600                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research NVIDIA in semiconductors and AI computing with financials, competitors, news, ratings. Include  │
│  sources.                                                                                                       │
│  ID: d8b8d830-58a2-477d-8043-a050de639797                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieved ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Completed                                                                                     │
│  Time: 1774.12ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investment Research Manager                                                                             │
│                                                                                                                 │
│  Task: Research NVIDIA in semiconductors and AI computing with financials, competitors, news, ratings. Include  │
│  sources.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#74) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'NVIDIA semiconductors AI computing financials competitors news ratings'}               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 16596.79ms                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 13832.82ms                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL OUTPUT
```markdown
# Investment Memo: NVIDIA Corporation

## Overview
NVIDIA Corporation is a leading player in the semiconductor industry, renowned for its advancements in artificial intelligence (AI) computing and graphics processing technology. The company holds a substantial market share in AI chips, positioning it well to benefit from the growing demand in this segment.

## Financial Performance
- **Revenue**: NVIDIA reported revenues of **$150.3 billion**, reflecting a **54% year-over-year increase**.
- **Net Income**: The company exhibited an impressive **80% increase** in net income, rising to **$22 billion**.
- **Future Guidance**: NVIDIA has projected first-quarter revenue guidance of **$43 billion**, surpassing market estimates, indicating strong momentum going into the next quarter.

## Market Position and Competitors
NVIDIA commands a dominant position within the AI chip sector, possessing a market share ranging from **70% to 95%** for AI accelerators. However, it i

In [74]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

llm = "gpt-4o-mini"

# =========================
# 🧠 MANAGER (BRAIN)
# =========================
manager = Agent(
    role="Chief Investment Officer (Manager)",
    goal=(
        "Deliver a high-quality investment memo by intelligently delegating tasks. "
        "First ensure research is completed, then validated, then writing, then review."
    ),
    backstory=(
        "You are a CIO managing a team of analysts. You ensure accuracy, clarity, "
        "and investment-grade output."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=True,
)

# =========================
# 🔍 RESEARCHER
# =========================
researcher = Agent(
    role="Equity Research Analyst",
    goal="Collect accurate financials, competitors, news, and analyst ratings with sources.",
    backstory="CFA with deep experience in equity markets.",
    llm=llm,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
)

# =========================
# ✅ VALIDATOR (NEW 🔥)
# =========================
validator = Agent(
    role="Financial Data Validator",
    goal="Verify all research facts and ensure every claim has a valid source.",
    backstory="Ex-auditor ensuring data integrity and correctness.",
    llm=llm,
    verbose=True,
)

# =========================
# ✍️ WRITER
# =========================
writer = Agent(
    role="Investment Memo Writer",
    goal="Write a clean, structured, executive-ready memo.",
    backstory="Former Bloomberg journalist.",
    llm=llm,
    verbose=True,
)

# =========================
# 🧾 REVIEWER (NEW 🔥)
# =========================
reviewer = Agent(
    role="Senior Investment Reviewer",
    goal="Improve clarity, remove fluff, and ensure professional tone.",
    backstory="Portfolio manager reviewing analyst reports.",
    llm=llm,
    verbose=True,
)

# =========================
# 📋 TASKS
# =========================
research_task = Task(
    description=(
        "Research {company_name} in {industry}.\n"
        "- Financials\n- Competitors\n- News\n- Analyst ratings\n"
        "Include source URLs."
    ),
    expected_output="Detailed bullet research with sources.",
    agent=researcher,
)

validation_task = Task(
    description=(
        "Validate the research:\n"
        "- Check accuracy\n"
        "- Ensure sources exist\n"
        "- Flag inconsistencies"
    ),
    expected_output="Validated research with corrections if needed.",
    agent=validator,
    context=[research_task],
)

writing_task = Task(
    description=(
        "Write investment memo with:\n"
        "- Executive Summary\n- Financials\n- Competition\n- Risks\n- Recommendation"
    ),
    expected_output="Structured Markdown memo.",
    agent=writer,
    context=[validation_task],
)

review_task = Task(
    description=(
        "Improve the memo:\n"
        "- Make concise\n"
        "- Improve tone\n"
        "- Ensure clarity"
    ),
    expected_output="Final polished memo.",
    agent=reviewer,
    context=[writing_task],
)

# =========================
# 🚀 CREW (HIERARCHICAL)
# =========================
research_crew = Crew(
    agents=[researcher, validator, writer, reviewer],  # workers only
    tasks=[research_task, validation_task, writing_task, review_task],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True,
)

# =========================
# ▶️ RUN
# =========================
print("\n📊 EXPECTED FLOW:")
print("Manager → Research → Validate → Write → Review → Output\n")

result = research_crew.kickoff(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)


📊 EXPECTED FLOW:
Manager → Research → Validate → Write → Review → Output



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 20f18934-62d7-423a-a1a9-28b10b1ece40                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research NVIDIA in semiconductors and AI computing.                                                      │
│  - Financials                                                                                                   │
│  - Competitors                                                                                                  │
│  - News                                                                                                         │
│  - Analyst ratings                                                                                              │
│  Include source URLs.                                                                                           │
│  ID: b4279d3e-3e95-4b62-b774-c23791079e14                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Investment Officer (Manager)                                                                      │
│                                                                                                                 │
│  Task: Research NVIDIA in semiconductors and AI computing.                                                      │
│  - Financials                                                                                                   │
│  - Competitors                                                                                                  │
│  - News                                                                                                         │
│  - Analyst ratings                                                                                              │
│  Include source URLs.                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#75) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'NVIDIA semiconductors financials 2023'}                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Improve the memo:                                                                                        │
│  - Make concise                                                                                                 │
│  - Improve tone                                                                                                 │
│  - Ensure clarity                                                                                               │
│  ID: 00256639-a77c-4ef3-ad4c-2cc903ffa631                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Investment Officer (Manager)                                                                      │
│                                                                                                                 │
│  Task: Improve the memo:                                                                                        │
│  - Make concise                                                                                                 │
│  - Improve tone                                                                                                 │
│  - Ensure clarity                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Investment Officer (Manager)                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```markdown                                                                                                    │
│  # NVIDIA Investment Memo                                                                                       │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│  NVIDIA Corporation stands as a preeminent force in the AI computing sector, demonstrating remarkable           │
│  financial growth in recent quarters. The company achieved substantial revenues, particularly in the Data       │
│  Center segment, despite facing formidable competition and external challenges. With a high analyst consensus   │
│  rating and a promising trajectory in the AI chip market, NVIDIA is set for sustained success, albeit with      │
│  risks linked to market dynamics and regulatory constraints.                                                    │
│                                                                                                                 │
│  ## Financials                                                                                                  │
│  - **Fiscal Year 2023 Revenue**: $26.97 billion, flat year-over-year.                                           │
│  [Source](https://www.sec.gov/Archives/edgar/data/1045810/000104581023000017/nvda-20230129.htm)                 │
│                                                                                                                 │
│  - **Recent Financial Results**:                                                                                │
│    - Q3 Fiscal 2024 revenue: $18.12 billion, a 206% year-over-year increase.                                    │
│  [Source](https://investor.nvidia.com/news/press-release-details/2023/NVIDIA-Announces-Financial-Results-for-T  │
│  hird-Quarter-Fiscal-2024/default.aspx)                                                                         │
│                                                                                                                 │
│    - Q4 Fiscal 2023 revenue: $6.051 billion, with a gross margin of 63.3%.                                      │
│  [Source](https://www.hpcwire.com/off-the-wire/nvidia-announces-financial-results-for-fourth-quarter-and-fisca  │
│  l-2023/)                                                                                                       │
│                                                                                                                 │
│  - **Data Center Growth**: For the latest fiscal quarter, Data Center revenue reached approximately $10.3       │
│  billion, demonstrating significant year-over-year growth, correcting the previously misreported figure of      │
│  $62.3 billion. [Source](https://investor.nvidia.com/financial-info/financial-reports/default.aspx)             │
│                                                                                                                 │
│  ## Competition                                                                                                 │
│  - **Key Competitors in AI Computing**:                                                                         │
│    - **AMD**: Recently launched MI300 chips for AI work

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Improve the memo:                                                                                        │
│  - Make concise                                                                                                 │
│  - Improve tone                                                                                                 │
│  - Ensure clarity                                                                                               │
│  Agent: Chief Investment Officer (Manager)                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL OUTPUT
```markdown
# NVIDIA Investment Memo

## Executive Summary
NVIDIA Corporation stands as a preeminent force in the AI computing sector, demonstrating remarkable financial growth in recent quarters. The company achieved substantial revenues, particularly in the Data Center segment, despite facing formidable competition and external challenges. With a high analyst consensus rating and a promising trajectory in the AI chip market, NVIDIA is set for sustained success, albeit with risks linked to market dynamics and regulatory constraints.

## Financials
- **Fiscal Year 2023 Revenue**: $26.97 billion, flat year-over-year. [Source](https://www.sec.gov/Archives/edgar/data/1045810/000104581023000017/nvda-20230129.htm)

- **Recent Financial Results**:
  - Q3 Fiscal 2024 revenue: $18.12 billion, a 206% year-over-year increase. [Source](https://investor.nvidia.com/news/press-release-details/2023/NVIDIA-Announces-Financial-Results-for-Third-Quarter-Fiscal-2024/default.aspx)
  
  - Q4 

In [75]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

llm = "gpt-4o-mini"

# =========================
# 🧠 PLANNER AGENT (NEW 🔥)
# =========================
planner = Agent(
    role="Execution Planner",
    goal=(
        "Create a step-by-step execution plan for producing an investment memo. "
        "Clearly define which agent should do what and in what order."
    ),
    backstory="Expert in breaking down complex workflows into clear execution steps.",
    llm=llm,
    verbose=True,
)

# =========================
# 🧠 MANAGER
# =========================
manager = Agent(
    role="Chief Investment Officer",
    goal=(
        "Execute the plan efficiently by delegating tasks to the right agents. "
        "Ensure high-quality output."
    ),
    backstory="Senior decision-maker managing analysts and writers.",
    llm=llm,
    verbose=True,
    allow_delegation=True,
)

# =========================
# 👨‍💻 WORKERS
# =========================
researcher = Agent(
    role="Equity Research Analyst",
    goal="Gather accurate financial and market data with sources.",
    backstory="CFA with deep research expertise.",
    llm=llm,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
)

validator = Agent(
    role="Data Validator",
    goal="Verify accuracy and consistency of research.",
    backstory="Audit specialist.",
    llm=llm,
    verbose=True,
)

writer = Agent(
    role="Investment Memo Writer",
    goal="Write structured and professional investment memo.",
    backstory="Financial journalist.",
    llm=llm,
    verbose=True,
)

reviewer = Agent(
    role="Senior Reviewer",
    goal="Refine and improve clarity and quality.",
    backstory="Portfolio manager.",
    llm=llm,
    verbose=True,
)

# =========================
# 📋 STEP 1: PLAN TASK
# =========================
planning_task = Task(
    description=(
        "Create a detailed execution plan to analyze {company_name} in {industry}.\n"
        "Include:\n"
        "- Step-by-step workflow\n"
        "- Which agent performs each step\n"
        "- Expected outputs\n"
        "Keep it structured and clear."
    ),
    expected_output="Step-by-step execution plan.",
    agent=planner,
)

# =========================
# 📋 STEP 2: EXECUTION TASKS
# =========================
research_task = Task(
    description="Perform research on {company_name} with financials, competitors, news.",
    expected_output="Research with sources.",
    agent=researcher,
)

validation_task = Task(
    description="Validate research for accuracy and completeness.",
    expected_output="Validated research.",
    agent=validator,
)

writing_task = Task(
    description="Write investment memo.",
    expected_output="Structured memo.",
    agent=writer,
)

review_task = Task(
    description="Refine and finalize memo.",
    expected_output="Final polished memo.",
    agent=reviewer,
)

# =========================
# 🧩 CREW 1 → PLANNING
# =========================
planning_crew = Crew(
    agents=[planner],
    tasks=[planning_task],
    process=Process.sequential,
    verbose=True,
)

# =========================
# ▶️ RUN PLANNING FIRST
# =========================
plan = planning_crew.kickoff(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("🧠 EXECUTION PLAN (BEFORE RUNNING AGENTS)")
print("=" * 60)
print(plan)

# =========================
# 🧩 CREW 2 → AUTONOMOUS EXECUTION
# =========================
execution_crew = Crew(
    agents=[researcher, validator, writer, reviewer],
    tasks=[research_task, validation_task, writing_task, review_task],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True,
)

# =========================
# ▶️ RUN EXECUTION
# =========================
result = execution_crew.kickoff(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("🚀 FINAL OUTPUT")
print("=" * 60)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2f313333-807a-4ebb-8553-9ad51ee4fa11                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Create a detailed execution plan to analyze NVIDIA in semiconductors and AI computing.                   │
│  Include:                                                                                                       │
│  - Step-by-step workflow                                                                                        │
│  - Which agent performs each step                                                                               │
│  - Expected outputs                                                                                             │
│  Keep it structured and clear.                                                                                  │
│  ID: b32cbd9a-940b-49d7-8313-56955bda38f1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Execution Planner                                                                                       │
│                                                                                                                 │
│  Task: Create a detailed execution plan to analyze NVIDIA in semiconductors and AI computing.                   │
│  Include:                                                                                                       │
│  - Step-by-step workflow                                                                                        │
│  - Which agent performs each step                                                                               │
│  - Expected outputs                                                                                             │
│  Keep it structured and clear.                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Execution Planner                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Step-by-Step Execution Plan for Analyzing NVIDIA in Semiconductors and AI Computing                         │
│                                                                                                                 │
│  ### Agent Roles                                                                                                │
│  1. **Research Analyst (RA)**                                                                                   │
│  2. **Financial Analyst (FA)**                                                                                  │
│  3. **Industry Specialist (IS)**                                                                                │
│  4. **Data Analyst (DA)**                                                                                       │
│  5. **Editor (E)**                                                                                              │
│  6. **Project Manager (PM)**                                                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Step 1: Gather Preliminary Information                                                                     │
│  - **Agent**: Research Analyst (RA)                                                                             │
│  - **Actions**:                                                                                                 │
│    1. Review existing literature and recent publications on NVIDIA and its role in semiconductors and AI.       │
│    2. Collect relevant data on NVIDIA's market position, competitors, and industry trends.                      │
│  - **Expected Outputs**: A summary report containing key findings, recent news articles, and an overview of     │
│  NVIDIA's market presence.                                                                                      │
│                                                                                                                 │
│  ### Step 2: Financial Performance Analysis                                                                     │
│  - **Agent**: Financial Analyst (FA)                                                                            │
│  - **Actions**:                                                                                                 │
│    1. Analyze NVIDIA's financial statements (income statement, balance sheet, cash flow).                       │
│    2. Calculate key financial ratios (P/E, EBITDA margin, ROE, etc.).                                           │
│    3. Compare these metrics with industry peers.                                                                │
│  - **Expected Outputs**: A financial analysis report highlighting NVIDIA's performance metrics compared to      │
│  competitors.                                                                                                   │
│                                                                                                                 │
│  ### Step 3: Deep Dive into Product and Technology     

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Create a detailed execution plan to analyze NVIDIA in semiconductors and AI computing.                   │
│  Include:                                                                                                       │
│  - Step-by-step workflow                                                                                        │
│  - Which agent performs each step                                                                               │
│  - Expected outputs                                                                                             │
│  Keep it structured and clear.                                                                                  │
│  Agent: Execution Planner                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Refine and finalize memo.                                                                                │
│  ID: 5f837bc9-9fee-4472-9e5a-963aee488b4e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Investment Officer                                                                                │
│                                                                                                                 │
│  Task: Refine and finalize memo.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Investment Officer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Investment Memo: NVIDIA Financials Q3 2023**                                                                 │
│                                                                                                                 │
│  **I. Executive Summary**                                                                                       │
│  NVIDIA has reported a remarkable performance in Q3 2023, achieving record revenue alongside significant        │
│  growth in earnings per share. This memo highlights the financial results, competitor landscape, and recent     │
│  developments that may impact NVIDIA's future growth and investment potential.                                  │
│                                                                                                                 │
│  **II. Financial Results**                                                                                      │
│  - NVIDIA reported revenue of $18.12 billion for Q3 2023, marking a 206% increase year-over-year and a 34%      │
│  increase from the previous quarter.                                                                            │
│  - The substantial growth is primarily driven by the surging demand for artificial intelligence (AI) and data   │
│  center solutions.                                                                                              │
│  - Earnings per share (EPS) reached $3.43, surpassing analysts’ expectations significantly, showcasing strong   │
│  operational performance.                                                                                       │
│  - For Q4, NVIDIA has provided guidance suggesting an approximate revenue of $20 billion, indicating sustained  │
│  momentum in its core markets.                                                                                  │
│                                                                                                                 │
│  **III. Additional Metrics**                                                                                    │
│  - Operating income has also shown robust growth, reinforcing NVIDIA's leading position in the AI chip market.  │
│  - Continued investment in AI and data center technologies indicates a strategic focus on high-growth           │
│  potential sectors.                                                                                             │
│                                                                                                                 │
│  **IV. Market Analysis**                                                                                        │
│  - NVIDIA currently commands approximately 80% of the AI chip market; however, competitive pressures are        │
│  increasing.                                                                                                    │
│  - Key competitors include:                                                                                     │
│    - **Advanced Micro Devices (AMD)**: Competing heavily in the graphics and processing unit sectors, AMD is    │
│  particularly focused on gaming and AI.                                                                         │
│    - **Intel**: Actively investing to secure a foothold in AI chip development, aiming to compete across        │
│  various segments.                                     

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Refine and finalize memo.                                                                                │
│  Agent: Chief Investment Officer                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


🚀 FINAL OUTPUT
**Investment Memo: NVIDIA Financials Q3 2023**

**I. Executive Summary**  
NVIDIA has reported a remarkable performance in Q3 2023, achieving record revenue alongside significant growth in earnings per share. This memo highlights the financial results, competitor landscape, and recent developments that may impact NVIDIA's future growth and investment potential.

**II. Financial Results**  
- NVIDIA reported revenue of $18.12 billion for Q3 2023, marking a 206% increase year-over-year and a 34% increase from the previous quarter.  
- The substantial growth is primarily driven by the surging demand for artificial intelligence (AI) and data center solutions.  
- Earnings per share (EPS) reached $3.43, surpassing analysts’ expectations significantly, showcasing strong operational performance.  
- For Q4, NVIDIA has provided guidance suggesting an approximate revenue of $20 billion, indicating sustained momentum in its core markets.

**III. Additional Metrics**  
- Operating

---
## 7. Implementation 1 — Research and Report Writing Crew

**Scenario:** An investment firm needs a quick market research report on a company before a decision meeting. Two agents collaborate: one researches, one writes.

---
## 8. Implementation 2 — Software Development Crew

**Scenario:** A startup needs a Python module built. A product manager, software engineer, and QA reviewer collaborate to produce production-ready code.

In [59]:
from crewai import Agent, Task, Crew, Process

# --- Agents ---
product_manager = Agent(
    role="Technical Product Manager",
    goal="Translate business requirements into precise technical specifications.",
    backstory=(
        "You have 8 years of experience as a PM at companies like Stripe and Twilio. "
        "You write specs that engineers can implement without asking follow-up questions."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

software_engineer = Agent(
    role="Senior Python Engineer",
    goal="Write clean, well-documented, production-quality Python code.",
    backstory=(
        "You are a principal engineer who has worked on backend systems processing "
        "millions of requests per day. You follow PEP 8, write docstrings, and always "
        "include type hints and error handling."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

qa_reviewer = Agent(
    role="QA Engineer",
    goal="Identify bugs, edge cases, and security issues in code before it ships.",
    backstory=(
        "You are a QA lead who has caught critical bugs that saved companies from "
        "production incidents. You are thorough and never approve code without "
        "checking for input validation, error handling, and at least 3 edge cases."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# --- Tasks ---
spec_task = Task(
    description=(
        "Write a technical specification for a Python function called "
        "`calculate_compound_interest`. "
        "Requirements: accepts principal (float), annual rate (float), "
        "years (int), and compounds_per_year (int, default 12). "
        "Returns final amount (float) rounded to 2 decimal places. "
        "Must handle invalid inputs gracefully."
    ),
    expected_output=(
        "A technical spec document including: function signature, parameter "
        "descriptions, return type, formula to use, and 3 input/output examples."
    ),
    agent=product_manager,
)

coding_task = Task(
    description=(
        "Implement the function described in the specification. "
        "Use type hints, a comprehensive docstring, raise ValueError for invalid inputs, "
        "and include 5 unit tests using pytest within the same file."
    ),
    expected_output=(
        "A complete, runnable Python file with the function implementation "
        "and pytest test suite."
    ),
    agent=software_engineer,
    context=[spec_task],
)

review_task = Task(
    description=(
        "Review the provided Python code for: (1) correctness of the compound interest "
        "formula, (2) completeness of input validation, (3) test coverage gaps, "
        "(4) any security or performance concerns. "
        "Provide a PASS or FAIL verdict with specific line-by-line comments."
    ),
    expected_output=(
        "A code review report with: verdict (PASS/FAIL), a list of issues found "
        "(if any), and the corrected code if changes are needed."
    ),
    agent=qa_reviewer,
    context=[coding_task],
    output_file="code_review_report.md",
)

# --- Crew ---
dev_crew = Crew(
    agents=[product_manager, software_engineer, qa_reviewer],
    tasks=[spec_task, coding_task, review_task],
    process=Process.sequential,
    verbose=True,
)

result = dev_crew.kickoff()

print("\n" + "=" * 60)
print("CODE REVIEW RESULT")
print("=" * 60)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 05566048-6552-4fcf-8e6d-4f09afd37b15                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Ending event 'memory_save_failed' emitted with empty scope stack. Missing starting 
event?

╭───────────────────────────────────────────── ❌ Memory Save Error ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Failed                                                                                             │
│  Source: Unified Memory                                                                                         │
│  Error: Error code: 403 - {'error': {'message': 'You are not allowed to generate embeddings from this model',   │
│  'type': 'invalid_request_error', 'param': None, 'code': None}}                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Write a technical specification for a Python function called `calculate_compound_interest`.              │
│  Requirements: accepts principal (float), annual rate (float), years (int), and compounds_per_year (int,        │
│  default 12). Returns final amount (float) rounded to 2 decimal places. Must handle invalid inputs gracefully.  │
│  ID: 60dbd0ec-3ed4-4fd3-b501-4f2e794b56e6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Product Manager                                                                               │
│                                                                                                                 │
│  Task: Write a technical specification for a Python function called `calculate_compound_interest`.              │
│  Requirements: accepts principal (float), annual rate (float), years (int), and compounds_per_year (int,        │
│  default 12). Returns final amount (float) rounded to 2 decimal places. Must handle invalid inputs gracefully.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Review the provided Python code for: (1) correctness of the compound interest formula, (2) completeness  │
│  of input validation, (3) test coverage gaps, (4) any security or performance concerns. Provide a PASS or FAIL  │
│  verdict with specific line-by-line comments.                                                                   │
│  ID: 905c9287-6dcf-4cb8-bf3e-b1da9c409b66                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer                                                                                             │
│                                                                                                                 │
│  Task: Review the provided Python code for: (1) correctness of the compound interest formula, (2) completeness  │
│  of input validation, (3) test coverage gaps, (4) any security or performance concerns. Provide a PASS or FAIL  │
│  verdict with specific line-by-line comments.                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: QA Engineer                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```python                                                                                                      │
│  Verdict: FAIL                                                                                                  │
│                                                                                                                 │
│  Issues Found:                                                                                                  │
│  1. **Correctness of the Compound Interest Formula**:                                                           │
│     - The implementation of the formula is correct. The formula `A = P * (1 + r/n)^(nt)` used in the code is    │
│  accurate for calculating compound interest.                                                                    │
│                                                                                                                 │
│  2. **Input Validation Completeness**:                                                                          │
│     - The validation checks are mostly thorough, but the function does not check if `years` is of type `int`.   │
│  Adding type checks can prevent unexpected behavior.                                                            │
│     - The function should explicitly handle cases where the user might pass non-numeric types, like strings or  │
│  None, to prevent type errors during the calculations.                                                          │
│                                                                                                                 │
│  3. **Test Coverage Gaps**:                                                                                     │
│     - There are no tests to check the exact output format (i.e., number of decimal places).                     │
│     - There should be additional edge cases tested, such as providing zero as a principal (should raise an      │
│  error), checking the smallest positive float value, or the largest reasonable float value.                     │
│                                                                                                                 │
│  4. **Security or Performance Concerns**:                                                                       │
│     - There are no immediate security concerns with the current implementation, but scaling should be           │
│  considered for very large numbers.                                                                             │
│                                                                                                                 │
│  ### Corrected Code                                                                                             │
│                                                                                                                 │
│  ```python                                                                                                      │
│  import pytest                                                                                                  │
│                                                                                                                 │
│  def calculate_compound_interest(principal: float, annual_rate: float, years: int, compounds_per_year: int =    │
│  12) -> float:                                         

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the provided Python code for: (1) correctness of the compound interest formula, (2) completeness  │
│  of input validation, (3) test coverage gaps, (4) any security or performance concerns. Provide a PASS or FAIL  │
│  verdict with specific line-by-line comments.                                                                   │
│  Agent: QA Engineer                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


CODE REVIEW RESULT
```python
Verdict: FAIL

Issues Found:
1. **Correctness of the Compound Interest Formula**:
   - The implementation of the formula is correct. The formula `A = P * (1 + r/n)^(nt)` used in the code is accurate for calculating compound interest.

2. **Input Validation Completeness**:
   - The validation checks are mostly thorough, but the function does not check if `years` is of type `int`. Adding type checks can prevent unexpected behavior.
   - The function should explicitly handle cases where the user might pass non-numeric types, like strings or None, to prevent type errors during the calculations.

3. **Test Coverage Gaps**:
   - There are no tests to check the exact output format (i.e., number of decimal places).
   - There should be additional edge cases tested, such as providing zero as a principal (should raise an error), checking the smallest positive float value, or the largest reasonable float value.

4. **Security or Performance Concerns**:
   - There are

---
## 9. Implementation 3 — Customer Support Triage Crew

**Scenario:** An e-commerce company receives customer complaints. Agents classify tickets, draft responses, and escalate high-priority issues.

In [61]:
from crewai import Agent, Task, Crew, Process
from pydantic import BaseModel
from typing import Literal

class TriageResult(BaseModel):
    ticket_id: str
    category: Literal["billing", "shipping", "product", "technical", "other"]
    priority: Literal["low", "medium", "high", "critical"]
    sentiment: Literal["positive", "neutral", "negative", "angry"]
    requires_escalation: bool
    summary: str

# --- Agents ---
triage_agent = Agent(
    role="Customer Support Triage Specialist",
    goal=(
        "Accurately classify customer tickets by category, priority, and sentiment "
        "so that the right team handles them with the right urgency."
    ),
    backstory=(
        "You managed the support inbox at a high-growth SaaS company for 5 years. "
        "You can instantly recognize whether a ticket is a billing dispute, a technical "
        "bug, or a shipping complaint, and you know exactly what makes a ticket critical."
    ),
    llm=llm,
    verbose=True,
)

response_agent = Agent(
    role="Senior Customer Success Manager",
    goal=(
        "Write empathetic, professional, and solution-focused responses that "
        "resolve customer issues on the first contact whenever possible."
    ),
    backstory=(
        "You have a background in psychology and customer experience. You hold a "
        "CSAT score of 4.9/5.0 across 10,000 tickets. Your responses always "
        "acknowledge the issue, apologize where appropriate, and provide a clear next step."
    ),
    llm=llm,
    verbose=True,
)

# --- Tasks ---
triage_task = Task(
    description=(
        "Analyze the following customer support ticket and classify it:\n\n"
        "Ticket ID: {ticket_id}\n"
        "Customer Message: {customer_message}\n\n"
        "Return structured output with: category, priority (critical if the customer "
        "mentions a chargeback, legal action, or data breach), sentiment, "
        "requires_escalation (True if critical or angry + high), and a 1-sentence summary."
    ),
    expected_output="A structured JSON object matching the TriageResult schema.",
    agent=triage_agent,
    output_pydantic=TriageResult,
)

response_task = Task(
    description=(
        "Using the triage classification, draft a customer-facing response email. "
        "Tone must match sentiment: warmer and more apologetic for negative/angry tickets. "
        "If requires_escalation is True, mention that a senior specialist will follow up "
        "within 2 business hours. Always include a ticket reference number."
    ),
    expected_output=(
        "A complete customer response email with subject line and body. "
        "Professional, empathetic, and under 200 words."
    ),
    agent=response_agent,
    context=[triage_task],
)

# --- Crew ---
support_crew = Crew(
    agents=[triage_agent, response_agent],
    tasks=[triage_task, response_task],
    process=Process.sequential,
    verbose=True,
    tracing=True,
)

# Simulate a customer complaint
result = support_crew.kickoff(inputs={
    "ticket_id": "TKT-20481",
    "customer_message": (
        "I ordered a laptop 3 weeks ago and it still has not arrived. "
        "The tracking link on your website has not updated in 10 days. "
        "This is completely unacceptable. I need this for work and if I don't "
        "receive it by tomorrow I will dispute the charge with my credit card company."
    )
})

print("\n" + "=" * 60)
print("SUPPORT CREW OUTPUT")
print("=" * 60)
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 216f556f-09a0-4e61-a88f-6611227ee9bb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following customer support ticket and classify it:                                           │
│                                                                                                                 │
│  Ticket ID: TKT-20481                                                                                           │
│  Customer Message: I ordered a laptop 3 weeks ago and it still has not arrived. The tracking link on your       │
│  website has not updated in 10 days. This is completely unacceptable. I need this for work and if I don't       │
│  receive it by tomorrow I will dispute the charge with my credit card company.                                  │
│                                                                                                                 │
│  Return structured output with: category, priority (critical if the customer mentions a chargeback, legal       │
│  action, or data breach), sentiment, requires_escalation (True if critical or angry + high), and a 1-sentence   │
│  summary.                                                                                                       │
│  ID: d10674b4-f9db-4700-a3f8-51c7d1ddd433                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Triage Specialist                                                                      │
│                                                                                                                 │
│  Task: Analyze the following customer support ticket and classify it:                                           │
│                                                                                                                 │
│  Ticket ID: TKT-20481                                                                                           │
│  Customer Message: I ordered a laptop 3 weeks ago and it still has not arrived. The tracking link on your       │
│  website has not updated in 10 days. This is completely unacceptable. I need this for work and if I don't       │
│  receive it by tomorrow I will dispute the charge with my credit card company.                                  │
│                                                                                                                 │
│  Return structured output with: category, priority (critical if the customer mentions a chargeback, legal       │
│  action, or data breach), sentiment, requires_escalation (True if critical or angry + high), and a 1-sentence   │
│  summary.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Triage Specialist                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ticket_id='TKT-20481' category='shipping' priority='critical' sentiment='angry' requires_escalation=True       │
│  summary='The customer is frustrated due to not receiving a laptop ordered 3 weeks ago and threatens to         │
│  dispute the charge if not resolved.'                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Analyze the following customer support ticket and classify it:                                           │
│                                                                                                                 │
│  Ticket ID: TKT-20481                                                                                           │
│  Customer Message: I ordered a laptop 3 weeks ago and it still has not arrived. The tracking link on your       │
│  website has not updated in 10 days. This is completely unacceptable. I need this for work and if I don't       │
│  receive it by tomorrow I will dispute the charge with my credit card company.                                  │
│                                                                                                                 │
│  Return structured output with: category, priority (critical if the customer mentions a chargeback, legal       │
│  action, or data breach), sentiment, requires_escalation (True if critical or angry + high), and a 1-sentence   │
│  summary.                                                                                                       │
│  Agent: Customer Support Triage Specialist                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Customer Success Manager                                                                         │
│                                                                                                                 │
│  Task: Using the triage classification, draft a customer-facing response email. Tone must match sentiment:      │
│  warmer and more apologetic for negative/angry tickets. If requires_escalation is True, mention that a senior   │
│  specialist will follow up within 2 business hours. Always include a ticket reference number.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Customer Success Manager                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Urgent Assistance Regarding Your Recent Order - TKT-20481                                             │
│                                                                                                                 │
│  Dear [Customer's Name],                                                                                        │
│                                                                                                                 │
│  I sincerely apologize for the frustration you’re experiencing regarding your laptop order. I understand how    │
│  concerning it is to wait for an item, especially when it's been three weeks. Please know that your             │
│  satisfaction is very important to us, and I appreciate your patience so far.                                   │
│                                                                                                                 │
│  I’ve escalated your case to our senior specialist, who will review the details and get back to you within the  │
│  next 2 business hours. We’re committed to resolving this matter quickly so you can receive your laptop as      │
│  soon as possible.                                                                                              │
│                                                                                                                 │
│  Thank you for bringing this to our attention, and again, I apologize for any inconvenience caused. If you      │
│  have any further questions or concerns in the meantime, please don’t hesitate to reach out.                    │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│  Senior Customer Success Manager                                                                                │
│  [Your Company]                                                                                                 │
│  Ticket Reference: TKT-20481                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the triage classification, draft a customer-facing response email. Tone must match sentiment:      │
│  warmer and more apologetic for negative/angry tickets. If requires_escalation is True, mention that a senior   │
│  specialist will follow up within 2 business hours. Always include a ticket reference number.                   │
│  Agent: Senior Customer Success Manager                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


SUPPORT CREW OUTPUT
Subject: Urgent Assistance Regarding Your Recent Order - TKT-20481

Dear [Customer's Name],

I sincerely apologize for the frustration you’re experiencing regarding your laptop order. I understand how concerning it is to wait for an item, especially when it's been three weeks. Please know that your satisfaction is very important to us, and I appreciate your patience so far.

I’ve escalated your case to our senior specialist, who will review the details and get back to you within the next 2 business hours. We’re committed to resolving this matter quickly so you can receive your laptop as soon as possible.

Thank you for bringing this to our attention, and again, I apologize for any inconvenience caused. If you have any further questions or concerns in the meantime, please don’t hesitate to reach out.

Best regards,

[Your Name]  
Senior Customer Success Manager  
[Your Company]  
Ticket Reference: TKT-20481


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 216f556f-09a0-4e61-a88f-6611227ee9bb                                                                       │
│  Final Output: Subject: Urgent Assistance Regarding Your Recent Order - TKT-20481                               │
│                                                                                                                 │
│  Dear [Customer's Name],                                                                                        │
│                                                                                                                 │
│  I sincerely apologize for the frustration you’re experiencing regarding your laptop order. I understand how    │
│  concerning it is to wait for an item, especially when it's been three weeks. Please know that your             │
│  satisfaction is very important to us, and I appreciate your patience so far.                                   │
│                                                                                                                 │
│  I’ve escalated your case to our senior specialist, who will review the details and get back to you within the  │
│  next 2 business hours. We’re committed to resolving this matter quickly so you can receive your laptop as      │
│  soon as possible.                                                                                              │
│                                                                                                                 │
│  Thank you for bringing this to our attention, and again, I apologize for any inconvenience caused. If you      │
│  have any further questions or concerns in the meantime, please don’t hesitate to reach out.                    │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│                                                                                                                 │
│  [Your Name]                                                                                                    │
│  Senior Customer Success Manager                                                                                │
│  [Your Company]                                                                                                 │
│  Ticket Reference: TKT-20481                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 10. Implementation 4 — Financial Analysis Crew with Memory

**Scenario:** A multi-turn financial analysis session where agents remember previous findings and build on them across multiple crew runs.

In [62]:
from crewai import Agent, Task, Crew, Process

# Memory-enabled crew persists knowledge between runs
# using short-term (in-run) and long-term (vector store) memory.

financial_analyst = Agent(
    role="Quantitative Financial Analyst",
    goal=(
        "Analyze financial metrics, identify trends, and compute key ratios "
        "to assess the financial health of companies."
    ),
    backstory=(
        "You hold a PhD in Financial Economics from Wharton and spent 10 years "
        "at a quant hedge fund. You think in numbers, ratios, and distributions."
    ),
    llm=llm,
    memory=True,
    verbose=True,
)

strategy_advisor = Agent(
    role="Corporate Strategy Advisor",
    goal=(
        "Translate financial analysis into actionable strategic recommendations "
        "that leadership teams can present to their boards."
    ),
    backstory=(
        "You are a former managing director at a Big Four consulting firm. "
        "You bridge the gap between financial data and strategic action."
    ),
    llm=llm,
    memory=True,
    verbose=True,
)

# Provide raw financial data as context in the task description
financial_data = """
Company: Acme Corp
FY2023 Revenue: $4.2B (up 18% YoY)
Gross Margin: 62%
Operating Income: $840M
Net Income: $610M
EPS: $3.82
Debt/Equity: 0.45
Current Ratio: 2.1
Free Cash Flow: $520M
R&D Spend: 14% of revenue
Employee Count: 18,400 (up 6% YoY)
"""

analysis_task = Task(
    description=(
        f"Analyze the following financial data for Acme Corp:\n{financial_data}\n"
        "Compute: P/E ratio context, revenue growth sustainability assessment, "
        "capital efficiency score, and flag any red flags or strengths. "
        "Compare ratios to industry medians for enterprise software companies."
    ),
    expected_output=(
        "A financial analysis report with computed ratios, trend commentary, "
        "3 key strengths, and 2 areas of concern."
    ),
    agent=financial_analyst,
)

strategy_task = Task(
    description=(
        "Based on the financial analysis, provide 3 strategic recommendations "
        "for Acme Corp's leadership team. Each recommendation must include: "
        "the strategic action, rationale grounded in the financial data, "
        "expected outcome, and a risk if not acted upon."
    ),
    expected_output=(
        "Three structured strategic recommendations formatted as a board-ready slide outline."
    ),
    agent=strategy_advisor,
    context=[analysis_task],
)

# Memory-enabled crew — uses text-embedding-3-small for vector memory
financial_crew = Crew(
    agents=[financial_analyst, strategy_advisor],
    tasks=[analysis_task, strategy_task],
    process=Process.sequential,
    memory=True,
    embedder={
        "provider": "openai",
        "config": {"model": "text-embedding-3-small"}
    },
    verbose=True,
)

result = financial_crew.kickoff()
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d66cf210-a8b9-4cfb-8be1-5844073d1fa7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following financial data for Acme Corp:                                                      │
│                                                                                                                 │
│  Company: Acme Corp                                                                                             │
│  FY2023 Revenue: $4.2B (up 18% YoY)                                                                             │
│  Gross Margin: 62%                                                                                              │
│  Operating Income: $840M                                                                                        │
│  Net Income: $610M                                                                                              │
│  EPS: $3.82                                                                                                     │
│  Debt/Equity: 0.45                                                                                              │
│  Current Ratio: 2.1                                                                                             │
│  Free Cash Flow: $520M                                                                                          │
│  R&D Spend: 14% of revenue                                                                                      │
│  Employee Count: 18,400 (up 6% YoY)                                                                             │
│                                                                                                                 │
│  Compute: P/E ratio context, revenue growth sustainability assessment, capital efficiency score, and flag any   │
│  red flags or strengths. Compare ratios to industry medians for enterprise software companies.                  │
│  ID: 578ce0b5-615c-40f6-95ac-1bc5755f23c8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ❌ Memory Query Error ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Query Failed                                                                                            │
│  Source: Unified Memory                                                                                         │
│  Error: Error code: 403 - {'error': {'message': 'You are not allowed to generate embeddings from this model',   │
│  'type': 'invalid_request_error', 'param': None, 'code': None}}                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Quantitative Financial Analyst                                                                          │
│                                                                                                                 │
│  Task: Analyze the following financial data for Acme Corp:                                                      │
│                                                                                                                 │
│  Company: Acme Corp                                                                                             │
│  FY2023 Revenue: $4.2B (up 18% YoY)                                                                             │
│  Gross Margin: 62%                                                                                              │
│  Operating Income: $840M                                                                                        │
│  Net Income: $610M                                                                                              │
│  EPS: $3.82                                                                                                     │
│  Debt/Equity: 0.45                                                                                              │
│  Current Ratio: 2.1                                                                                             │
│  Free Cash Flow: $520M                                                                                          │
│  R&D Spend: 14% of revenue                                                                                      │
│  Employee Count: 18,400 (up 6% YoY)                                                                             │
│                                                                                                                 │
│  Compute: P/E ratio context, revenue growth sustainability assessment, capital efficiency score, and flag any   │
│  red flags or strengths. Compare ratios to industry medians for enterprise software companies.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['industry medians for enterprise software companies', 'P/E ratio for enterprise software',  │
│  'financial analysis criteria']}                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Corporate Strategy Advisor                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Strategic Recommendations for Acme Corp's Leadership Team                                                    │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Recommendation 1: Optimize R&D Spending                                                                    │
│                                                                                                                 │
│  **Strategic Action:**                                                                                          │
│  - Reassess and optimize R&D investments to focus on high-impact projects that promise significant returns.     │
│                                                                                                                 │
│  **Rationale:**                                                                                                 │
│  - Acme Corp's R&D spending represents 14% of its revenue, which, while necessary for growth, poses a risk to   │
│  overall margin sustainability. With growing pressures to convert R&D efforts into market share, a more         │
│  targeted approach will enhance profitability.                                                                  │
│                                                                                                                 │
│  **Expected Outcome:**                                                                                          │
│  - By concentrating on projects with the highest potential for returns, Acme Corp can improve margins, driving  │
│  profitability without sacrificing growth initiatives.                                                          │
│                                                                                                                 │
│  **Risk if Not Acted Upon:**                                                                                    │
│  - Failure to optimize R&D spending may lead to diminishing returns on investments, resulting in eroded profit  │
│  margins and unsustainable growth in a competitive environment.                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Recommendation 2: Diversify Revenue Streams                                                                │
│                                                                                                                 │
│  **Strategic Action:**                                                                                          │
│  - Develop and implement a strategy to diversify Acme Corp's revenue streams through innovative product         │
│  offerings and strategic partnerships.                                                                          │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the financial analysis, provide 3 strategic recommendations for Acme Corp's leadership team.    │
│  Each recommendation must include: the strategic action, rationale grounded in the financial data, expected     │
│  outcome, and a risk if not acted upon.                                                                         │
│  Agent: Corporate Strategy Advisor                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d66cf210-a8b9-4cfb-8be1-5844073d1fa7                                                                       │
│  Final Output: # Strategic Recommendations for Acme Corp's Leadership Team                                      │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Recommendation 1: Optimize R&D Spending                                                                    │
│                                                                                                                 │
│  **Strategic Action:**                                                                                          │
│  - Reassess and optimize R&D investments to focus on high-impact projects that promise significant returns.     │
│                                                                                                                 │
│  **Rationale:**                                                                                                 │
│  - Acme Corp's R&D spending represents 14% of its revenue, which, while necessary for growth, poses a risk to   │
│  overall margin sustainability. With growing pressures to convert R&D efforts into market share, a more         │
│  targeted approach will enhance profitability.                                                                  │
│                                                                                                                 │
│  **Expected Outcome:**                                                                                          │
│  - By concentrating on projects with the highest potential for returns, Acme Corp can improve margins, driving  │
│  profitability without sacrificing growth initiatives.                                                          │
│                                                                                                                 │
│  **Risk if Not Acted Upon:**                                                                                    │
│  - Failure to optimize R&D spending may lead to diminishing returns on investments, resulting in eroded profit  │
│  margins and unsustainable growth in a competitive environment.                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Recommendation 2: Diversify Revenue Streams                                                                │
│                                                                                                                 │
│  **Strategic Action:**                                                                                          │
│  - Develop and implement a strategy to diversify Acme Corp's revenue streams through innovative product         │
│  offerings and strategic partnerships.                                                                          │
│                                                       

# Strategic Recommendations for Acme Corp's Leadership Team

---

### Recommendation 1: Optimize R&D Spending

**Strategic Action:**
- Reassess and optimize R&D investments to focus on high-impact projects that promise significant returns.

**Rationale:**
- Acme Corp's R&D spending represents 14% of its revenue, which, while necessary for growth, poses a risk to overall margin sustainability. With growing pressures to convert R&D efforts into market share, a more targeted approach will enhance profitability.

**Expected Outcome:**
- By concentrating on projects with the highest potential for returns, Acme Corp can improve margins, driving profitability without sacrificing growth initiatives.

**Risk if Not Acted Upon:**
- Failure to optimize R&D spending may lead to diminishing returns on investments, resulting in eroded profit margins and unsustainable growth in a competitive environment.

---

### Recommendation 2: Diversify Revenue Streams

**Strategic Action:**
- Develop and implem



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

---
## 11. Implementation 5 — RAG-Powered Knowledge Base Crew

**Scenario:** A legal team wants agents to answer questions grounded in their internal policy documents using Retrieval-Augmented Generation (RAG).

In [71]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import PDFSearchTool, TXTSearchTool

# PDFSearchTool uses text-embedding-3-small under the hood
# to chunk, embed, and semantically retrieve relevant passages
# from the provided document before passing them to the agent.

# Assumes you have a PDF file at this path
policy_tool = PDFSearchTool(
    pdf="company_policy.pdf",
    config={
        "llm": {
            "provider": "openai",
            "config": {"model": "gpt-4o-mini", "temperature": 0.0}
        },
        "embedder": {
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"}
        }
    }
)

legal_researcher = Agent(
    role="Corporate Legal Researcher",
    goal=(
        "Answer legal and compliance questions accurately by retrieving "
        "relevant clauses from internal policy documents."
    ),
    backstory=(
        "You are a paralegal at a Fortune 500 company specializing in "
        "corporate compliance and employment law. You always cite the "
        "specific section of the policy document in your answers."
    ),
    llm=llm,
    tools=[policy_tool],
    verbose=True,
)

rag_task = Task(
    description=(
        "Answer the following employee question using only the content of "
        "the company policy document. Do not speculate beyond what the "
        "document states. Question: {employee_question}"
    ),
    expected_output=(
        "A direct answer to the question, followed by the exact policy section "
        "that supports the answer, and a note if the policy is silent on the topic."
    ),
    agent=legal_researcher,
)

rag_crew = Crew(
    agents=[legal_researcher],
    tasks=[rag_task],
    process=Process.sequential,
    verbose=True,
)

# Example query
result = rag_crew.kickoff(inputs={
    "employee_question": (
        "Am I entitled to paid parental leave if I have been with the company "
        "for 6 months, and if so, how many weeks?"
    )
})

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7bed3ea9-8fe1-4337-936f-fb91de3a8970                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer the following employee question using only the content of the company policy document. Do not     │
│  speculate beyond what the document states. Question: Am I entitled to paid parental leave if I have been with  │
│  the company for 6 months, and if so, how many weeks?                                                           │
│  ID: 65de8688-3fb0-4e8c-823f-ce91e8f0a79c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Corporate Legal Researcher                                                                              │
│                                                                                                                 │
│  Task: Answer the following employee question using only the content of the company policy document. Do not     │
│  speculate beyond what the document states. Question: Am I entitled to paid parental leave if I have been with  │
│  the company for 6 months, and if so, how many weeks?                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_a_pdfs_content                                                                                    │
│  Args: {'query': 'paid parental leave'}                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_a_pdfs_content executed with result: Relevant Content:
No relevant content found....


...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_a_pdfs_content                                                                                    │
│  Output: Relevant Content:                                                                                      │
│  No relevant content found.                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Corporate Legal Researcher                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The policy is silent on the topic of paid parental leave.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer the following employee question using only the content of the company policy document. Do not     │
│  speculate beyond what the document states. Question: Am I entitled to paid parental leave if I have been with  │
│  the company for 6 months, and if so, how many weeks?                                                           │
│  Agent: Corporate Legal Researcher                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The policy is silent on the topic of paid parental leave.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7bed3ea9-8fe1-4337-936f-fb91de3a8970                                                                       │
│  Final Output: The policy is silent on the topic of paid parental leave.                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 12. Advanced: Custom Tools

You can build any custom tool by subclassing `BaseTool` or using the `@tool` decorator.

In [70]:
from crewai.tools import BaseTool, tool
import requests
from pydantic import Field

# --- Method 1: @tool decorator (for simple functions) ---
@tool("Currency Converter")
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Converts a monetary amount from one currency to another.
    Uses a live exchange rate API.
    Example: currency_converter(100, 'USD', 'EUR')
    """
    # In production, use a real API like frankfurter.app
    # This is a placeholder showing the pattern
    url = f"https://api.frankfurter.app/latest?amount={amount}&from={from_currency}&to={to_currency}"
    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        converted = data["rates"][to_currency]
        return f"{amount} {from_currency} = {converted:.2f} {to_currency}"
    except Exception as e:
        return f"Error fetching exchange rate: {str(e)}"


# --- Method 2: BaseTool subclass (for complex tools with state) ---
class DatabaseLookupTool(BaseTool):
    name: str = "Customer Database Lookup"
    description: str = (
        "Looks up customer information from the internal database "
        "by customer ID. Returns account status, plan, and join date."
    )
    # Simulate an in-memory database
    database: dict = Field(default_factory=lambda: {
        "CUST-001": {"name": "Apex Industries", "plan": "Enterprise", "status": "Active", "since": "2021-03-15"},
        "CUST-002": {"name": "Nova Retail Group", "plan": "Pro", "status": "Past Due", "since": "2022-08-22"},
        "CUST-003": {"name": "Meridian Healthcare", "plan": "Starter", "status": "Active", "since": "2023-11-01"},
    })

    def _run(self, customer_id: str) -> str:
        record = self.database.get(customer_id.upper())
        if record:
            return (
                f"Customer: {record['name']} | Plan: {record['plan']} | "
                f"Status: {record['status']} | Customer since: {record['since']}"
            )
        return f"No customer found with ID: {customer_id}"


# Use the custom tools in an agent
billing_agent = Agent(
    role="Billing Support Specialist",
    goal="Resolve billing disputes by looking up account details and performing currency calculations.",
    backstory="You handle billing inquiries and account status checks for a SaaS platform.",
    llm=llm,
    tools=[DatabaseLookupTool(), currency_converter],
    verbose=True,
)

billing_task = Task(
    description=(
        "Look up customer CUST-002 and determine their account status. "
        "If their account is past due, calculate what their monthly Pro plan fee of "
        "$299 USD would be in EUR and GBP for their international billing team."
    ),
    expected_output=(
        "Account status summary with currency conversions and a recommended action."
    ),
    agent=billing_agent,
)

billing_crew = Crew(
    agents=[billing_agent],
    tasks=[billing_task],
    process=Process.sequential,
    verbose=True,
)

result = billing_crew.kickoff()
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0d8a462b-5656-40d2-a076-826b1a04eb8f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Look up customer CUST-002 and determine their account status. If their account is past due, calculate    │
│  what their monthly Pro plan fee of $299 USD would be in EUR and GBP for their international billing team.      │
│  ID: 33f3d54f-2935-4a9a-9baa-32783ee08021                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Billing Support Specialist                                                                              │
│                                                                                                                 │
│  Task: Look up customer CUST-002 and determine their account status. If their account is past due, calculate    │
│  what their monthly Pro plan fee of $299 USD would be in EUR and GBP for their international billing team.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool customer_database_lookup executed with result: Customer: Nova Retail Group | Plan: Pro | Status: Past Due | Customer since: 2022-08-22...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: customer_database_lookup                                                                                 │
│  Args: {'customer_id': 'CUST-002'}                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Billing Support Specialist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Account Status Summary for Nova Retail Group**                                                               │
│                                                                                                                 │
│  - **Account Status:** Past Due                                                                                 │
│  - **Plan:** Pro                                                                                                │
│  - **Customer Since:** 2022-08-22                                                                               │
│                                                                                                                 │
│  **Monthly Fee in International Currencies:**                                                                   │
│  - **EUR:** €255.56                                                                                             │
│  - **GBP:** £220.69                                                                                             │
│                                                                                                                 │
│  **Recommended Action:** Immediate payment is advised to avoid further penalties and ensure continued service.  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Look up customer CUST-002 and determine their account status. If their account is past due, calculate    │
│  what their monthly Pro plan fee of $299 USD would be in EUR and GBP for their international billing team.      │
│  Agent: Billing Support Specialist                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0d8a462b-5656-40d2-a076-826b1a04eb8f                                                                       │
│  Final Output: **Account Status Summary for Nova Retail Group**                                                 │
│                                                                                                                 │
│  - **Account Status:** Past Due                                                                                 │
│  - **Plan:** Pro                                                                                                │
│  - **Customer Since:** 2022-08-22                                                                               │
│                                                                                                                 │
│  **Monthly Fee in International Currencies:**                                                                   │
│  - **EUR:** €255.56                                                                                             │
│  - **GBP:** £220.69                                                                                             │
│                                                                                                                 │
│  **Recommended Action:** Immediate payment is advised to avoid further penalties and ensure continued service.  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Account Status Summary for Nova Retail Group**

- **Account Status:** Past Due
- **Plan:** Pro
- **Customer Since:** 2022-08-22

**Monthly Fee in International Currencies:**
- **EUR:** €255.56
- **GBP:** £220.69

**Recommended Action:** Immediate payment is advised to avoid further penalties and ensure continued service.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 13. Advanced: Human-in-the-Loop

In [76]:
from crewai import Agent, Task, Crew, Process

# When human_input=True on a task, CrewAI will pause execution
# after the agent produces its draft output and prompt the user
# to review and optionally provide correction before finalizing.

legal_drafter = Agent(
    role="Legal Contract Drafter",
    goal="Draft legally sound contract clauses based on provided parameters.",
    backstory=(
        "You are a contract attorney specializing in SaaS and technology agreements. "
        "Your clauses are clear, enforceable, and client-protective."
    ),
    llm=llm,
    verbose=True,
)

draft_task = Task(
    description=(
        "Draft a limitation of liability clause for a SaaS agreement where: "
        "- Liability cap is 12 months of fees paid"
        "- Excludes gross negligence and willful misconduct from the cap"
        "- Mutual limitation (applies to both vendor and customer)"
        "- Governed by Delaware law"
    ),
    expected_output="A formal, numbered legal clause ready for insertion into a contract.",
    agent=legal_drafter,
    human_input=True,   # Agent pauses here for human review before finalizing
)

# Note: Running this cell will pause execution and prompt for input
# Uncomment to run:
legal_crew = Crew(
    agents=[legal_drafter],
    tasks=[draft_task],
    process=Process.sequential,
    verbose=True,
)
result = legal_crew.kickoff()
print(result)

print("Human-in-the-loop task configured. Uncomment the crew kickoff to run interactively.")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c36059e8-2687-4b7a-9d4e-32071443b1c4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Draft a limitation of liability clause for a SaaS agreement where: - Liability cap is 12 months of fees  │
│  paid- Excludes gross negligence and willful misconduct from the cap- Mutual limitation (applies to both        │
│  vendor and customer)- Governed by Delaware law                                                                 │
│  ID: 5ebd4bb8-a117-41ee-9ff7-509ac3c02487                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Contract Drafter                                                                                  │
│                                                                                                                 │
│  Task: Draft a limitation of liability clause for a SaaS agreement where: - Liability cap is 12 months of fees  │
│  paid- Excludes gross negligence and willful misconduct from the cap- Mutual limitation (applies to both        │
│  vendor and customer)- Governed by Delaware law                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Contract Drafter                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Limitation of Liability**                                                                                    │
│                                                                                                                 │
│  1. **Limitation of Liability**. Except for Claims arising from gross negligence or willful misconduct, in no   │
│  event shall either party’s total liability to the other party arising out of or in connection with this        │
│  Agreement exceed the total fees paid by the Customer to the Vendor under this Agreement in the twelve (12)     │
│  months preceding the event giving rise to the liability. This limitation of liability shall apply to all       │
│  claims, regardless of the form of action, whether in contract, tort (including negligence), or otherwise.      │
│                                                                                                                 │
│  2. **Mutual Exclusion**. The limitation of liability set forth in Section 1 shall apply mutually to both the   │
│  Vendor and the Customer, and each party acknowledges that the other party would not enter into this Agreement  │
│  without such limitation.                                                                                       │
│                                                                                                                 │
│  3. **Governing Law**. This Limitation of Liability clause shall be governed by and construed in accordance     │
│  with the laws of the State of Delaware, without regard to its conflict of laws principles.                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── 💬 Human Feedback Required ───────────────────────────────────────────╮
│                                                                                                                 │
│  Provide feedback on the Final Result above.                                                                    │
│                                                                                                                 │
│  • If you are happy with the result, simply hit Enter without typing anything.                                  │
│  • Otherwise, provide specific improvement requests.                                                            │
│  • You can provide multiple rounds of feedback until satisfied.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Draft a limitation of liability clause for a SaaS agreement where: - Liability cap is 12 months of fees  │
│  paid- Excludes gross negligence and willful misconduct from the cap- Mutual limitation (applies to both        │
│  vendor and customer)- Governed by Delaware law                                                                 │
│  Agent: Legal Contract Drafter                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c36059e8-2687-4b7a-9d4e-32071443b1c4                                                                       │
│  Final Output: **Limitation of Liability**                                                                      │
│                                                                                                                 │
│  1. **Limitation of Liability**. Except for Claims arising from gross negligence or willful misconduct, in no   │
│  event shall either party’s total liability to the other party arising out of or in connection with this        │
│  Agreement exceed the total fees paid by the Customer to the Vendor under this Agreement in the twelve (12)     │
│  months preceding the event giving rise to the liability. This limitation of liability shall apply to all       │
│  claims, regardless of the form of action, whether in contract, tort (including negligence), or otherwise.      │
│                                                                                                                 │
│  2. **Mutual Exclusion**. The limitation of liability set forth in Section 1 shall apply mutually to both the   │
│  Vendor and the Customer, and each party acknowledges that the other party would not enter into this Agreement  │
│  without such limitation.                                                                                       │
│                                                                                                                 │
│  3. **Governing Law**. This Limitation of Liability clause shall be governed by and construed in accordance     │
│  with the laws of the State of Delaware, without regard to its conflict of laws principles.                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Limitation of Liability**

1. **Limitation of Liability**. Except for Claims arising from gross negligence or willful misconduct, in no event shall either party’s total liability to the other party arising out of or in connection with this Agreement exceed the total fees paid by the Customer to the Vendor under this Agreement in the twelve (12) months preceding the event giving rise to the liability. This limitation of liability shall apply to all claims, regardless of the form of action, whether in contract, tort (including negligence), or otherwise.

2. **Mutual Exclusion**. The limitation of liability set forth in Section 1 shall apply mutually to both the Vendor and the Customer, and each party acknowledges that the other party would not enter into this Agreement without such limitation.

3. **Governing Law**. This Limitation of Liability clause shall be governed by and construed in accordance with the laws of the State of Delaware, without regard to its conflict of laws principl



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

---
## 14. Advanced: Async Execution and Callbacks

In [77]:
from crewai import Agent, Task, Crew, Process
from datetime import datetime

# --- Step Callback: Called after every agent action ---
def step_callback(agent_action):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] STEP: {agent_action}")

# --- Task Callback: Called when a task completes ---
def task_callback(task_output):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] TASK COMPLETED. Output length: {len(str(task_output))} chars")

# --- Async Kickoff: Non-blocking execution ---
async def run_crew_async():
    summarizer = Agent(
        role="Executive Summarizer",
        goal="Produce concise executive summaries of lengthy documents.",
        backstory="You distill complex content into key decisions and action items.",
        llm=llm,
    )

    summary_task = Task(
        description=(
            "Summarize the following meeting notes into 5 bullet points, "
            "each focused on an action item with an owner and deadline:\n\n"
            "Meeting: Q3 Product Roadmap Review\n"
            "Sarah: We need to ship the new dashboard by end of October or we lose the Acme contract.\n"
            "Tom: The API team is blocked on authentication. They need a decision on OAuth vs SAML by Friday.\n"
            "Sarah: Marketing wants a demo environment set up by September 15 for the conference.\n"
            "James: We agreed to deprecate v1 API on November 1. We need a migration guide published by October 1.\n"
            "Tom: Budget approval for the new infrastructure is needed from Finance before September 30."
        ),
        expected_output="5 action items, each with: task, owner, and deadline.",
        agent=summarizer,
        callback=task_callback,
    )

    crew = Crew(
        agents=[summarizer],
        tasks=[summary_task],
        process=Process.sequential,
        step_callback=step_callback,
        verbose=True,
    )

    # kickoff_async returns a coroutine; await it in an async context
    result = await crew.kickoff_async()
    return result

# In a Jupyter notebook, use await directly
import asyncio
result = await run_crew_async()
print("\nFINAL OUTPUT:")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1b3a411c-f452-485d-8eae-19ed9d7f6b89                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Summarize the following meeting notes into 5 bullet points, each focused on an action item with an       │
│  owner and deadline:                                                                                            │
│                                                                                                                 │
│  Meeting: Q3 Product Roadmap Review                                                                             │
│  Sarah: We need to ship the new dashboard by end of October or we lose the Acme contract.                       │
│  Tom: The API team is blocked on authentication. They need a decision on OAuth vs SAML by Friday.               │
│  Sarah: Marketing wants a demo environment set up by September 15 for the conference.                           │
│  James: We agreed to deprecate v1 API on November 1. We need a migration guide published by October 1.          │
│  Tom: Budget approval for the new infrastructure is needed from Finance before September 30.                    │
│  ID: 7e885285-d739-4dc0-a0de-d29fe3bc88f6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Executive Summarizer                                                                                    │
│                                                                                                                 │
│  Task: Summarize the following meeting notes into 5 bullet points, each focused on an action item with an       │
│  owner and deadline:                                                                                            │
│                                                                                                                 │
│  Meeting: Q3 Product Roadmap Review                                                                             │
│  Sarah: We need to ship the new dashboard by end of October or we lose the Acme contract.                       │
│  Tom: The API team is blocked on authentication. They need a decision on OAuth vs SAML by Friday.               │
│  Sarah: Marketing wants a demo environment set up by September 15 for the conference.                           │
│  James: We agreed to deprecate v1 API on November 1. We need a migration guide published by October 1.          │
│  Tom: Budget approval for the new infrastructure is needed from Finance before September 30.                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06:25:36] STEP: AgentFinish(thought='Failed to parse LLM response', output='1. Task: Ship the new dashboard  \n   Owner: Sarah  \n   Deadline: End of October  \n\n2. Task: Make a decision on OAuth vs SAML  \n   Owner: Tom  \n   Deadline: Friday  \n\n3. Task: Set up a demo environment for marketing  \n   Owner: Sarah  \n   Deadline: September 15  \n\n4. Task: Publish a migration guide for deprecating v1 API  \n   Owner: James  \n   Deadline: October 1  \n\n5. Task: Obtain budget approval for new infrastructure from Finance  \n   Owner: Tom  \n   Deadline: Before September 30  ', text='1. Task: Ship the new dashboard  \n   Owner: Sarah  \n   Deadline: End of October  \n\n2. Task: Make a decision on OAuth vs SAML  \n   Owner: Tom  \n   Deadline: Friday  \n\n3. Task: Set up a demo environment for marketing  \n   Owner: Sarah  \n   Deadline: September 15  \n\n4. Task: Publish a migration guide for deprecating v1 API  \n   Owner: James  \n   Deadline: October 1  \n\n5. Task: Obtain budget a

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Executive Summarizer                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. Task: Ship the new dashboard                                                                                │
│     Owner: Sarah                                                                                                │
│     Deadline: End of October                                                                                    │
│                                                                                                                 │
│  2. Task: Make a decision on OAuth vs SAML                                                                      │
│     Owner: Tom                                                                                                  │
│     Deadline: Friday                                                                                            │
│                                                                                                                 │
│  3. Task: Set up a demo environment for marketing                                                               │
│     Owner: Sarah                                                                                                │
│     Deadline: September 15                                                                                      │
│                                                                                                                 │
│  4. Task: Publish a migration guide for deprecating v1 API                                                      │
│     Owner: James                                                                                                │
│     Deadline: October 1                                                                                         │
│                                                                                                                 │
│  5. Task: Obtain budget approval for new infrastructure from Finance                                            │
│     Owner: Tom                                                                                                  │
│     Deadline: Before September 30                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[06:25:36] TASK COMPLETED. Output length: 488 chars


╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Summarize the following meeting notes into 5 bullet points, each focused on an action item with an       │
│  owner and deadline:                                                                                            │
│                                                                                                                 │
│  Meeting: Q3 Product Roadmap Review                                                                             │
│  Sarah: We need to ship the new dashboard by end of October or we lose the Acme contract.                       │
│  Tom: The API team is blocked on authentication. They need a decision on OAuth vs SAML by Friday.               │
│  Sarah: Marketing wants a demo environment set up by September 15 for the conference.                           │
│  James: We agreed to deprecate v1 API on November 1. We need a migration guide published by October 1.          │
│  Tom: Budget approval for the new infrastructure is needed from Finance before September 30.                    │
│  Agent: Executive Summarizer                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL OUTPUT:
1. Task: Ship the new dashboard  
   Owner: Sarah  
   Deadline: End of October  

2. Task: Make a decision on OAuth vs SAML  
   Owner: Tom  
   Deadline: Friday  

3. Task: Set up a demo environment for marketing  
   Owner: Sarah  
   Deadline: September 15  

4. Task: Publish a migration guide for deprecating v1 API  
   Owner: James  
   Deadline: October 1  

5. Task: Obtain budget approval for new infrastructure from Finance  
   Owner: Tom  
   Deadline: Before September 30  


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1b3a411c-f452-485d-8eae-19ed9d7f6b89                                                                       │
│  Final Output: 1. Task: Ship the new dashboard                                                                  │
│     Owner: Sarah                                                                                                │
│     Deadline: End of October                                                                                    │
│                                                                                                                 │
│  2. Task: Make a decision on OAuth vs SAML                                                                      │
│     Owner: Tom                                                                                                  │
│     Deadline: Friday                                                                                            │
│                                                                                                                 │
│  3. Task: Set up a demo environment for marketing                                                               │
│     Owner: Sarah                                                                                                │
│     Deadline: September 15                                                                                      │
│                                                                                                                 │
│  4. Task: Publish a migration guide for deprecating v1 API                                                      │
│     Owner: James                                                                                                │
│     Deadline: October 1                                                                                         │
│                                                                                                                 │
│  5. Task: Obtain budget approval for new infrastructure from Finance                                            │
│     Owner: Tom                                                                                                  │
│     Deadline: Before September 30                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## 15. Best Practices and Common Pitfalls

### Best Practices

**Agent Design**
- Give each agent a single, clear responsibility. An agent trying to do too many things produces mediocre output for each.
- Write backstories that are specific and credible. Vague backstories produce generic responses.
- Set `allow_delegation=False` on specialized agents to prevent uncontrolled delegation chains.

**Task Design**
- Use numbered requirements in task descriptions. Agents are more likely to cover every point.
- The `expected_output` field acts as a quality rubric — the more specific, the better the output.
- Use `output_pydantic` when downstream code needs to consume structured data.

**Crew Design**
- Default to `Process.sequential` unless you need dynamic orchestration.
- Enable `memory=True` with `text-embedding-3-small` when agents need to reference earlier context.
- Use `max_rpm` at the crew level to avoid OpenAI rate limit errors in large crews.

### Common Pitfalls

| Pitfall | Cause | Fix |
|---|---|---|
| Agent produces off-topic output | Backstory or goal too vague | Make backstory more specific and role-constraining |
| Infinite tool loops | `max_iter` too high | Set `max_iter=8` or lower for most tasks |
| Context not flowing between tasks | Missing `context=[prev_task]` | Explicitly link dependent tasks via `context` |
| Pydantic validation errors | LLM output format inconsistent | Add format examples to `expected_output` |
| Rate limit errors on large crews | Too many simultaneous API calls | Set `max_rpm` at agent and crew level |